# Lección 3.6 — Mínimos locales, mesetas y por qué no siempre importan

> **Módulo 3 · Cómo aprende un modelo** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 90–110 min (lección ≈ 60 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 3.3 (descenso de gradiente) y Lección 3.5 (el ruido de SGD y mini-batch).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Reconocer los tres "terrenos traicioneros" del paisaje de la pérdida: **mínimos locales**, **mesetas** y **puntos de silla**.
- Distinguir un problema **convexo** (un solo valle) de uno **no convexo** (muchos valles).
- Aplicar tres remedios: **varios puntos de inicio**, **ruido** y **momento** (*momentum*).
- Explicar por qué, en la práctica, los mínimos locales preocupan **menos de lo que parece**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar ni matemáticas. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: los terrenos traicioneros

Nuestro excursionista en la niebla (Lección 3.3) solo sabe una cosa: **la inclinación bajo sus pies**. Cuando el suelo está plano, se detiene, convencido de haber llegado. Pero hay tres lugares donde el suelo está plano **y no es el refugio**:

```
  1. MÍNIMO LOCAL              2. MESETA                     3. PUNTO DE SILLA
  (una hondonada)              (un desierto plano)           (un paso de montaña)

  ╲        ╱‾╲                 ‾‾‾‾‾‾‾‾‾‾‾‾‾╲                  ⛰️  ↑ sube  ⛰️
   ╲  🧍  ╱   ╲                  🧍            ╲               ← baja  🧍  baja →
    ╲__╱       ╲      ╱                         ╲___🏠            ⛰️  ↓ sube  ⛰️
                ╲____╱ 🏠
  "Llegué"... pero hay         "No siento pendiente":       Plano justo en el centro,
  un valle más hondo.          avanza lentísimo.            pero baja hacia los lados.
```

- **Mínimo local:** un valle que **no es el más profundo**. El más profundo de todos se llama **mínimo global**.
- **Meseta** (*plateau*): una zona casi plana; la pendiente es diminuta y los pasos también.
- **Punto de silla** (*saddle point*): como el centro de una silla de montar: plano, pero baja en una dirección y sube en otra.

### ✋ Ejercicio 1
En los tres lugares, la pendiente es (casi) **cero**. Con lo que sabes de la regla del descenso de gradiente (`perilla = perilla − tasa × pendiente`), ¿qué hace el excursionista en cada uno? ¿Puede distinguir si está en el refugio o en una trampa?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con pendiente ≈ 0, el paso `tasa × pendiente` es ≈ 0: **se detiene o avanza muy lento** en los tres casos. Y **no puede distinguirlos** del refugio de verdad: solo siente su pendiente local, no la profundidad de los otros valles. Por eso necesitamos trucos para salir de estas trampas.
</details>

---
## 2. 📖 El concepto: paisajes con uno o con muchos valles

Hasta ahora todos nuestros paisajes eran un **tazón**: un solo valle. A eso se le llama un problema **convexo**. En un problema convexo, **cualquier** camino cuesta abajo llega al mismo fondo: no hay trampas.

- ✅ **Convexos:** regresión lineal con MSE, regresión logística con entropía cruzada, SVM.
- ⚠️ **No convexos:** redes neuronales (Módulo 9), K-means (Lección 8.2), muchos modelos con perillas que interactúan.

Construyamos un paisaje **no convexo** de juguete, con una sola perilla `w` y dos valles:

In [ ]:
def perdida(w):
    return 0.5 * w**4 - 2 * w**2 + 0.6 * w + 3

def pendiente(w):
    return 2 * w**3 - 4 * w + 0.6

ws = np.linspace(-2.3, 2.3, 300)
plt.plot(ws, perdida(ws), lw=2, color="gray")
plt.annotate("mínimo GLOBAL\n(el refugio)", (-1.48, perdida(-1.48)), xytext=(-2.2, 3.5), arrowprops=dict(arrowstyle="->"))
plt.annotate("mínimo LOCAL\n(una hondonada)", (1.33, perdida(1.33)), xytext=(0.9, 4.5), arrowprops=dict(arrowstyle="->"))
plt.xlabel("Perilla w"); plt.ylabel("Pérdida"); plt.title("Un paisaje con dos valles"); plt.grid(alpha=.3); plt.show()
print(f"Pérdida en el mínimo global (w ≈ -1.48): {perdida(-1.48):.2f}")
print(f"Pérdida en el mínimo local  (w ≈ +1.33): {perdida(1.33):.2f}")

### 🔮 Ejercicio 2 — ¿Adónde llega cada excursionista?
Soltamos excursionistas en distintos puntos: `w = -2`, `w = -0.5`, `w = 0.1`, `w = 0.3`, `w = 1` y `w = 2`. Todos usan descenso de gradiente normal. Mira la curva y predice en qué valle termina cada uno.

💡 Fíjate dónde está la "loma" que separa los dos valles.

✍️ *Mis predicciones:* -2 → ___ · -0.5 → ___ · 0.1 → ___ · 0.3 → ___ · 1 → ___ · 2 → ___

In [ ]:
def bajar(w, tasa=0.05, pasos=200, ruido=0.0, momento=0.0, semilla=0):
    rng = np.random.default_rng(semilla)
    camino, velocidad = [w], 0.0
    for paso in range(pasos):
        g = pendiente(w) + ruido * rng.normal() * (1 - paso / pasos)   # ruido opcional que se apaga poco a poco
        velocidad = momento * velocidad - tasa * g                      # momento opcional (Sección 3)
        w = w + velocidad
        camino.append(w)
    return np.array(camino)

plt.plot(ws, perdida(ws), lw=2, color="gray")
for inicio in [-2, -0.5, 0.1, 0.3, 1, 2]:
    c = bajar(inicio)
    plt.plot(c, perdida(c), ".-", ms=4, alpha=.7, label=f"inicio {inicio:>4} → termina en {c[-1]:+.2f}")
plt.xlabel("Perilla w"); plt.ylabel("Pérdida"); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- `-2`, `-0.5` y `0.1` → llegan al **mínimo global** (w ≈ −1.48).
- `0.3`, `1` y `2` → quedan atrapados en el **mínimo local** (w ≈ +1.33).

La frontera es la cima de la loma, en **w ≈ 0.15**: quien empieza a su izquierda cae al valle bueno, y quien empieza a su derecha cae en la hondonada. **El resultado depende del punto de partida.** Con un paisaje convexo eso nunca pasa.
</details>

---
## 3. 📖 Tres remedios para escapar

### Remedio 1: probar varios puntos de partida (*reinicios aleatorios*)
Si el resultado depende de dónde empiezas, **empieza en varios sitios** y quédate con el mejor final. Es barato de entender y muy usado: K-means lo hace con su hiperparámetro `n_init` (Lección 8.2), y en redes neuronales se hace cambiando la semilla (`random_state`).

### ✋ Ejercicio 3
Lanzamos **5 excursionistas** desde puntos al azar entre −2 y 2. ¿Cuál es la probabilidad aproximada de que **al menos uno** caiga en el valle bueno? (Pista: el valle bueno ocupa la parte de −2 a 0.15.)

✍️ *Tu respuesta:*

In [ ]:
N_INICIOS = 5          # ✏️ prueba 1, 2, 10
rng_inicios = np.random.default_rng(3)
inicios = rng_inicios.uniform(-2, 2, N_INICIOS)
finales = [bajar(w0)[-1] for w0 in inicios]
for w0, wf in zip(inicios, finales):
    print(f"inicio {w0:+.2f} → final {wf:+.2f} (pérdida {perdida(wf):.2f})")
mejor = min(finales, key=perdida)
print(f"\n🏆 Me quedo con w = {mejor:+.2f}, pérdida = {perdida(mejor):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El valle bueno ocupa ≈ 2.15 de los 4 unidades del rango (≈ 54 %). La probabilidad de que **los 5** caigan en la hondonada es ≈ 0.46⁵ ≈ 2 %, así que hay un **≈ 98 %** de que al menos uno encuentre el refugio. En esta ejecución, 3 de los 5 cayeron en el valle bueno.

Con pocos valles, unos cuantos reinicios bastan. Con **muchos** valles estrechos (lo verás en la práctica integradora), hacen falta muchos más.
</details>

### Remedio 2: un poco de ruido
En la Lección 3.5 viste que SGD y mini-batch **tiemblan**. Ese temblor, que parecía un defecto, puede **sacarte de una hondonada**: un empujón al azar en el momento justo te lleva por encima de la loma.

Hagamos el experimento: dejamos a 20 excursionistas **ya atrapados** en el mínimo local (`w = 1.33`) y les damos pasos con ruido que se va apagando poco a poco (como el decaimiento de la tasa de la Lección 3.5).

### 🔮 Ejercicio 4
Sin ruido, ¿cuántos de los 20 escapan? ¿Y con ruido 6? Haz tu apuesta y luego ✏️ prueba `RUIDO = 3` y `RUIDO = 10`.

✍️ *Tu respuesta:*

In [ ]:
RUIDO = 6        # ✏️
escapan = 0
for semilla in range(20):
    c = bajar(1.33, ruido=RUIDO, pasos=300, semilla=semilla)
    escapan += c[-1] < 0
    plt.plot(c, alpha=.5)
plt.axhline(-1.48, color="green", ls="--", label="mínimo global"); plt.axhline(1.33, color="red", ls="--", label="mínimo local")
plt.xlabel("Paso"); plt.ylabel("w"); plt.title(f"Ruido = {RUIDO}: {escapan} de 20 escaparon"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sin ruido** (0): ninguno escapa. Ya están en un punto con pendiente cero: no se mueven.
- Con **ruido 6**: **11 de 20** escapan al valle global. Los demás reciben empujones, pero vuelven a caer en la hondonada.
- Con ruido 3 no escapa ninguno (empujones demasiado débiles); con ruido 10 escapan más, pero el camino es mucho más caótico.

Por eso se dice que el ruido de los mini-lotes es un **aliado**: ayuda a explorar el paisaje. Y por eso conviene que **se apague** al final: para que el excursionista se asiente en el valle que encontró.
</details>

### Remedio 3: momento (*momentum*), la bola que rueda
En vez de un excursionista que se detiene en cuanto el suelo está plano, imagina una **bola pesada** que rueda: acumula **velocidad** en las bajadas y esa inercia puede llevarla **por encima** de una loma pequeña.

En la fórmula, cada paso suma un poco de la velocidad anterior: `velocidad = momento × velocidad − tasa × pendiente`. Con `momento = 0` es el descenso de gradiente normal; con `momento = 0.9` la bola "recuerda" el 90 % de su velocidad.

### 🔮 Ejercicio 5
Las bolas salen desde `w = 2`, `w = 2.5` y desde el propio mínimo local (`w = 1.33`), con momento 0, 0.7 y 0.9. ¿Cuáles crees que llegan al valle bueno? ¿El momento lo **garantiza**?

✍️ *Tu respuesta:*

In [ ]:
for momento in [0.0, 0.7, 0.9]:          # ✏️
    resultados = []
    for inicio in [1.33, 2.0, 2.5]:
        final = bajar(inicio, momento=momento)[-1]
        resultados.append(f"inicio {inicio} → {'✅ global' if final < 0 else '❌ local '}")
    print(f"momento = {momento}:   " + "   ·   ".join(resultados))

<details><summary><b>👀 Ver respuesta</b></summary>

- **Momento 0:** las tres bolas acaban en el mínimo local.
- **Momento 0.7:** la que sale de `w = 2.5` cruza la loma y llega al valle global; las otras dos no.
- **Momento 0.9:** ahora la que sale de `w = 2` cruza… ¡pero la de `w = 2.5` **no**! Con tanta velocidad pasa de largo, rebota y termina en la hondonada.
- La que empieza **quieta** en el mínimo local nunca escapa: sin pendiente no hay velocidad que acumular.

El momento **ayuda, pero no garantiza nada**: depende del terreno y del punto de partida. Aun así, los optimizadores modernos lo usan casi siempre (SGD con momento, **Adam**), sobre todo porque acelera el avance por cañones y mesetas. Los verás en la **Lección 9.5**.
</details>

---
## 4. 📖 Qué pasa si… el excursionista llega a una meseta

Una meseta no es una trampa permanente: hay pendiente, pero **diminuta**. El excursionista avanza a paso de hormiga y, desde fuera, **parece que el entrenamiento está roto**.

Este paisaje tiene un valle al que se llega cruzando un desierto casi plano:

In [ ]:
def perdida_meseta(w):
    return 1 - np.exp(-(w - 3) ** 2 / 2)

def pendiente_meseta(w):
    return (w - 3) * np.exp(-(w - 3) ** 2 / 2)

w, historia = -1.0, []
for paso in range(400):
    w = w - 1.0 * pendiente_meseta(w)
    historia.append(perdida_meseta(w))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
wm = np.linspace(-2, 6, 300)
a1.plot(wm, perdida_meseta(wm), lw=2, color="gray"); a1.plot(-1, perdida_meseta(-1), "ks", ms=9, label="Inicio (w = −1)")
a1.set_xlabel("Perilla w"); a1.set_ylabel("Pérdida"); a1.set_title("El paisaje: una meseta antes del valle"); a1.legend(); a1.grid(alpha=.3)
a2.plot(historia, lw=2); a2.set_xlabel("Paso"); a2.set_ylabel("Pérdida"); a2.set_title("La curva de pérdida"); a2.grid(alpha=.3)
plt.show()

### 🔮 Ejercicio 6 — ¿Lo habrías detenido?
Imagina que solo ves la curva de la derecha y que la estás mirando en el **paso 150**.
1. ¿Habrías pensado que el entrenamiento estaba roto?
2. ¿En qué paso, aproximadamente, la pérdida "despierta" y cae?
3. ¿Qué podrías haber hecho para cruzar la meseta más rápido?

✍️ *Tu respuesta:*

In [ ]:
historia = np.array(historia)
print(f"Pérdida en el paso 0: {historia[0]:.3f} · paso 100: {historia[100]:.3f} · paso 150: {historia[150]:.3f}")
print(f"La pérdida baja de 0.5 por primera vez en el paso {np.argmax(historia < 0.5)}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Muy probablemente **sí**: durante ~150 pasos la pérdida está pegada en ≈ 1.0 y parece que no aprende nada.
2. Despierta de golpe hacia el paso **220** (la pérdida baja de 0.5 por primera vez en el paso 222) y en pocos pasos más cae casi a 0.
3. Opciones: **más paciencia** (más épocas), una **tasa más alta** (prueba `2.0` en vez de `1.0` en la celda de arriba: cruza en la mitad de pasos), **momento** (la bola acelera en el plano) o un **mejor punto de partida** (inicialización, Lección 9.7).

⚠️ Lección práctica: **una curva plana al principio no siempre significa un error**. Antes de tirar un entrenamiento, revisa si la pérdida se mueve aunque sea un poquito.
</details>

---
## 5. 📖 El punto de silla: plano en el centro, cuesta abajo a los lados

Con **dos** perillas aparece una trampa nueva. Mira este paisaje: en la dirección `x` es un valle, pero en la dirección `y` es una loma. El centro `(0, 0)` es un **punto de silla**: allí la pendiente es cero en las dos direcciones, pero no es un mínimo. Los mínimos de verdad están en `y = ±1.41`.

### 🔮 Ejercicio 7
Soltamos dos excursionistas desde `x = 2`: uno **exactamente** en `y = 0` y otro en `y = 0.01` (un pelo desviado). ¿Qué crees que le pasa a cada uno?

✍️ *Tu respuesta:*

In [ ]:
def silla(x, y):
    return x**2 - y**2 + y**4 / 4

def gradiente_silla(x, y):
    return 2 * x, -2 * y + y**3

X, Y = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.2, 2.2, 200))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
a1.contourf(X, Y, silla(X, Y), levels=30, cmap="viridis")
for y0, color in [(0.0, "red"), (0.01, "orange")]:
    x, y, camino, perd = 2.0, y0, [], []
    for _ in range(60):
        gx, gy = gradiente_silla(x, y)
        x, y = x - 0.1 * gx, y - 0.1 * gy
        camino.append((x, y)); perd.append(silla(x, y))
    camino = np.array(camino)
    a1.plot(camino[:, 0], camino[:, 1], ".-", color=color, label=f"inicio (2, {y0})")
    a2.plot(perd, color=color, lw=2, label=f"inicio (2, {y0}) → termina en ({x:.2f}, {y:.2f})")
a1.plot(0, 0, "wx", ms=12, mew=3); a1.set_title("Paisaje con punto de silla (✕)"); a1.set_xlabel("x"); a1.set_ylabel("y"); a1.legend()
a2.set_xlabel("Paso"); a2.set_ylabel("Pérdida"); a2.set_title("Curvas de pérdida"); a2.legend(); a2.grid(alpha=.3)
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- El de **`y = 0` exacto** baja por la dirección `x` hasta el centro y **se queda en el punto de silla para siempre** (pérdida 0, cuando el fondo real está en −1). En `y = 0` la pendiente en `y` es exactamente cero: nada lo empuja hacia los lados.
- El de **`y = 0.01`** también llega cerca del centro, **se queda un rato** (la curva naranja hace una pequeña meseta) y luego se desliza hacia un mínimo real en `y ≈ 1.41`, con pérdida −1.

En la práctica, con miles o millones de perillas, es casi imposible estar *exactamente* en la línea de la silla, y el **ruido** de los mini-lotes ayuda a salir. Por eso los puntos de silla suelen causar **mesetas temporales** más que trampas definitivas.
</details>

---
## 6. 📖 ¿Por qué no siempre importan?

Con tantas trampas, ¿cómo es posible que las redes neuronales funcionen? Tres razones:

1. **Muchos modelos clásicos son convexos.** Regresión lineal y logística tienen un solo valle: no hay mínimos locales de los que preocuparse.
2. **En muchas dimensiones, casi todo lo "plano" son sillas, no hondonadas.** Para ser un mínimo local, el terreno tiene que subir en **todas** las direcciones a la vez. Con un millón de perillas, lo normal es que en alguna dirección todavía se pueda bajar, y el ruido la encuentra.
3. **En modelos grandes, los valles que se encuentran suelen ser casi igual de buenos.** No hace falta *el* mejor valle: basta con uno suficientemente bueno.

Comprobemos el punto 3 con redes neuronales de verdad (`MLPRegressor` de scikit-learn, Módulo 9). Entrenamos **10 veces** cada red, cambiando solo la semilla (el punto de partida), y medimos qué tan bien predice datos nuevos (R², Lección 4.1: 1 es perfecto).

### 🔮 Ejercicio 8 — ¿Red chica o red grande?
Comparamos una red **diminuta** (3 neuronas) con una **grande** (100 neuronas). ¿Cuál crees que dará resultados más **parecidos entre sí** al cambiar la semilla? (La celda tarda unos segundos.)

✍️ *Tu respuesta:*

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

rng_d = np.random.default_rng(0)
x_d = rng_d.uniform(-3, 3, (600, 1))
y_d = np.sin(2 * x_d[:, 0]) + 0.3 * x_d[:, 0] + rng_d.normal(0, 0.15, 600)
x_tr, x_te, y_tr, y_te = train_test_split(x_d, y_d, random_state=0)

for neuronas in [3, 100]:
    r2 = []
    for semilla in range(10):
        red = MLPRegressor(hidden_layer_sizes=(neuronas,), activation="tanh", learning_rate_init=0.01,
                           max_iter=1000, random_state=semilla).fit(x_tr, y_tr)
        r2.append(red.score(x_te, y_te))
    print(f"Red de {neuronas:>3} neuronas → R² en prueba con 10 semillas: {np.round(r2, 2)}")
    print(f"                       peor = {min(r2):.2f} · mejor = {max(r2):.2f} · diferencia = {max(r2) - min(r2):.2f}\n")

<details><summary><b>👀 Ver respuesta</b></summary>

La red **grande** es mucho más **estable**: con las 10 semillas su R² queda entre ≈ 0.86 y ≈ 0.90 (diferencia ≈ 0.04). La red **diminuta** varía bastante más: la mayoría ronda 0.85–0.90, pero con una semilla se queda en **≈ 0.67**: cayó en un valle claramente peor.

Esto ilustra una idea respaldada por la investigación: en redes grandes, los distintos valles a los que llegas suelen ser **igual de buenos**. Los mínimos locales "malos" son sobre todo un problema de modelos pequeños o mal inicializados.

📌 Consejo práctico: si los resultados cambian mucho con la semilla, **prueba varias semillas** (remedio 1) y desconfía de un modelo que solo funciona con una.
</details>

---
## 7. 📖 Tabla de trampas y perillas

| Trampa | Síntoma en la curva de pérdida | Remedios |
|---|---|---|
| **Mínimo local** | Se estanca en un valor peor que el de otras ejecuciones; distintas semillas dan resultados muy distintos | Varios inicios, ruido (mini-batch), momento, modelo más grande |
| **Meseta** | Plana durante mucho tiempo y luego cae de golpe | Paciencia, tasa más alta, momento/Adam, mejor inicialización |
| **Punto de silla** | Meseta temporal, a veces con "escalones" | Ruido (mini-batch), momento |

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| **Semilla / inicialización** (`random_state`) | Desde dónde empieza a caminar | — (no es "más o menos") | — | Prueba **varias** y compara |
| **Número de reinicios** (`n_init` en K-means) | Cuántas veces se entrena desde cero | Más probable encontrar un buen valle; más tiempo | Más rápido; más riesgo de quedar en una hondonada | 10 (K-means); 3–5 semillas en redes |
| **Momento** (`momentum`, `beta1` en Adam) | Cuánta "inercia" conserva | Cruza lomas y mesetas, pero puede pasarse de largo | Se comporta como descenso normal | 0.9 |
| **Tamaño de lote** | Cuánto ruido hay | Menos ruido: explora menos | Más ruido: explora más | 32–256 (Lección 3.5) |

## 8. 📖 Errores comunes y señales de alerta

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| Matar el entrenamiento a la primera meseta | Podía estar a punto de "despertar" | Mira si la pérdida se mueve aunque sea un poco; prueba con más épocas |
| Reportar el resultado de **una sola** semilla | Pudo ser suerte (buena o mala) | Entrena con 3–5 semillas y reporta el promedio y la variación |
| Preocuparse por mínimos locales en una regresión lineal | Es convexa: no tiene | Concéntrate en escalar y en la tasa |
| Pensar que el momento siempre ayuda | Puede pasarse de largo | Úsalo con valores estándar (0.9) y vigila la curva |

### 🔮 Ejercicio 9
Tu regresión logística (Lección 5.2) da resultados muy distintos cada vez que la entrenas con otra semilla. ¿Es culpa de los mínimos locales? ¿Qué revisarías?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** La regresión logística con entropía cruzada es **convexa**: tiene un solo valle. Si los resultados cambian mucho con la semilla, el problema está en otro lado: probablemente **no termina de converger** (pocas épocas, tasa mal elegida, datos sin escalar: Lecciones 3.3–3.4) o la semilla está cambiando **la división de los datos** (en cuyo caso conviene validación cruzada, Lección 4.6).
</details>

---
## 9. 🧠 Resumen en 5 puntos
1. El descenso de gradiente se detiene donde el suelo está plano: eso incluye **mínimos locales**, **mesetas** y **puntos de silla**, no solo el mínimo global.
2. En problemas **convexos** (regresión lineal, logística) solo hay un valle: no hay trampas. En **no convexos** (redes, K-means) el resultado depende del punto de partida.
3. Tres remedios: **varios inicios**, **ruido** (mini-batch) y **momento**. Ninguno es garantía; combinados funcionan muy bien.
4. Una **meseta** no es un error: a veces solo hace falta paciencia o una tasa mayor.
5. En modelos **grandes**, los valles que se encuentran suelen ser casi igual de buenos: por eso los mínimos locales importan **menos de lo que parece**.

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Qué tiene de especial un problema convexo?
- a) Que es muy lento · b) Que tiene un solo valle: cualquier camino cuesta abajo llega al mismo fondo · c) Que no necesita tasa de aprendizaje · d) Que no tiene pérdida

**2.** La curva de pérdida está plana durante 200 épocas y de repente cae. Lo más probable es que el modelo…
- a) haya cruzado una meseta · b) haya tenido un error en el código · c) esté sobreajustando · d) haya encontrado un mínimo local

**3.** ¿Cuál de estos remedios usa K-means a través de `n_init`?
- a) Momento · b) Ruido · c) Varios inicios · d) Bajar la tasa

**4.** Entrenas la misma red con 5 semillas y obtienes exactitudes de 91 %, 90 %, 92 %, 91 % y 63 %. ¿Qué pasó probablemente con la de 63 %?
- a) Sobreajuste · b) Quedó atrapada en un mal valle (o meseta) · c) Fuga de datos · d) Nada raro

**5.** ¿Por qué el ruido de mini-batch puede ayudar con los mínimos locales?
- a) Porque reduce la pérdida a cero · b) Porque los empujones al azar pueden sacar al modelo de una hondonada · c) Porque evita calcular el gradiente · d) No ayuda

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)**  2. **a)** El patrón "plana y luego cae" es la firma de una meseta (o un punto de silla).  3. **c)**  4. **b)** Las otras cuatro coinciden; una ejecución muy peor sugiere un mal punto de partida. Si pasa a menudo, revisa la inicialización y la tasa.  5. **b)**
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un amigo dice: *"Si el descenso de gradiente puede quedarse en mínimos locales, es un método defectuoso; deberíamos usar siempre búsqueda exhaustiva"*. Usa lo que aprendiste en la Lección 3.1 (la explosión de combinaciones) para responderle.

**B.** ¿Por qué crees que entrenar un modelo con **5 semillas distintas** y reportar el promedio es más honesto que reportar solo la mejor?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** La búsqueda exhaustiva es **imposible** con muchas perillas: con 100 perillas y 10 valores cada una serían 10¹⁰⁰ combinaciones. El descenso de gradiente es el único método práctico a esa escala, y con los remedios (inicios, ruido, momento) y modelos grandes, los valles que encuentra suelen ser suficientemente buenos. "Perfecto pero imposible" pierde contra "muy bueno y factible".

**B.** Porque la mejor de 5 incluye **suerte**: con datos nuevos o en producción no sabrás qué semilla "te tocó". El promedio (y la variación) dice qué esperar **de verdad** del método. Es la misma idea que no elegir el modelo mirando el conjunto de prueba (Lección 4.6).
</details>

### 10.3 🧩 Práctica integradora: el ritmo de las mareas 🌊

Un puerto mide cada media hora, durante 2 días, la **altura del agua** respecto a su nivel medio. Las mareas suben y bajan como una onda. Quieres descubrir **cada cuántas horas se repite** la marea.

El modelo: `altura = 1.5 × sin(f × hora)`. La altura máxima (1.5 m) ya la conoces; solo falta **una perilla**, `f`, la "velocidad" de la onda. El periodo en horas es `2π / f`.

**Paso 1 ▶️ — Los datos**

In [ ]:
rng_m = np.random.default_rng(4)
hora = np.arange(0, 48, 0.5)
altura = 1.5 * np.sin(0.507 * hora) + rng_m.normal(0, 0.3, len(hora))   # (el modelo NO conoce el 0.507)

def perdida_marea(f):
    return np.mean((1.5 * np.sin(f * hora) - altura) ** 2)

def pendiente_marea(f):
    return np.mean(2 * (1.5 * np.sin(f * hora) - altura) * 1.5 * hora * np.cos(f * hora))

plt.plot(hora, altura, ".", label="Mediciones"); plt.xlabel("Hora"); plt.ylabel("Altura (m)"); plt.grid(alpha=.3); plt.legend(); plt.show()

**Paso 2 🔮 — El paisaje.** Antes de ejecutar: ¿crees que el paisaje de la pérdida en función de `f` será un tazón (un solo valle) o tendrá muchos valles?

✍️ *Mi apuesta:*

In [ ]:
fs = np.linspace(0.05, 1.2, 1000)
plt.plot(fs, [perdida_marea(f) for f in fs], lw=1.5)
plt.xlabel("Perilla f"); plt.ylabel("Pérdida (MSE)"); plt.title("Paisaje de la pérdida de las mareas"); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

¡**Muchísimos valles**! Hay un valle muy profundo y estrecho (cerca de `f ≈ 0.5`, pérdida ≈ 0.09) y una fila de hondonadas poco profundas (pérdida ≈ 2) a ambos lados. Es un paisaje **no convexo** de libro: cada `f` "equivocada" encaja con los datos en algunos tramos y falla en otros.
</details>

**Paso 3 ✏️ — Un solo excursionista.** Usa descenso de gradiente desde `F_INICIO = 0.3`. ¿Dónde termina? ¿Es un buen modelo? Prueba también 0.45 y 0.8.

In [ ]:
def bajar_marea(f, tasa=0.0003, pasos=500):
    for _ in range(pasos):
        f = f - tasa * pendiente_marea(f)
    return f

F_INICIO = 0.3          # ✏️
f_final = bajar_marea(F_INICIO)
print(f"inicio f = {F_INICIO} → final f = {f_final:.3f} · pérdida = {perdida_marea(f_final):.3f} · periodo = {2 * np.pi / f_final:.1f} h")

<details><summary><b>👀 Ver respuesta</b></summary>

- Desde **0.3** termina en `f ≈ 0.349` (periodo ≈ 18 h), con pérdida ≈ **2.07**: atrapado en una hondonada. El modelo es pésimo.
- Desde **0.45** llega a `f ≈ 0.507` (periodo ≈ **12.4 h**), con pérdida ≈ 0.09: ¡el valle bueno!
- Desde **0.8** se queda en `f ≈ 0.799` (pérdida ≈ 2.13): otra hondonada.

Solo los inicios cercanos (aproximadamente entre 0.42 y 0.59) encuentran el valle bueno.
</details>

**Paso 4 ▶️ — Remedio 1 con pocos intentos.** Probamos **10 inicios al azar** entre 0.05 y 1.2. ¿Alguno encuentra el valle bueno?

In [ ]:
N_INICIOS = 10          # ✏️ (lo cambiarás en el Paso 5)
rng_i = np.random.default_rng(0)
inicios = rng_i.uniform(0.05, 1.2, N_INICIOS)
finales = np.array([bajar_marea(f0) for f0 in inicios])
perdidas = np.array([perdida_marea(f) for f in finales])
print("Pérdidas finales:", np.round(np.sort(perdidas), 2))
mejor = finales[np.argmin(perdidas)]
print(f"Mejor: f = {mejor:.3f} · pérdida = {perdidas.min():.3f} · periodo = {2 * np.pi / mejor:.1f} h")

<details><summary><b>👀 Ver respuesta</b></summary>

**Ninguno** de los 10 cae en el valle bueno: todas las pérdidas quedan alrededor de 2. El valle bueno ocupa solo ≈ 15 % del rango (de 0.42 a 0.59), así que la probabilidad de fallar 10 veces seguidas es ≈ 0.85¹⁰ ≈ 20 %. Mala suerte… pero ocurre.
</details>

**Paso 5 ✏️ — Más intentos.** Sube `N_INICIOS` en la celda anterior (prueba 30 y 50) y vuelve a ejecutarla. ¿Con cuántos aparece el valle bueno?

✍️ *Mi resultado:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con **30** inicios ya aparece (2 de los 30 lo encuentran: pérdida ≈ 0.09, periodo ≈ **12.4 horas**), y con 50, 7 lo encuentran. Cuantos más valles estrechos tiene el paisaje, **más reinicios** hacen falta.

💡 Truco profesional: como aquí solo hay **una** perilla, también puedes hacer primero una **búsqueda burda en cuadrícula** (como el gráfico del Paso 2) para encontrar la zona prometedora y luego **refinar** con descenso de gradiente. Con millones de perillas no se puede, pero con 1 o 2 es una gran estrategia.
</details>

**Paso 6 ✋ — Reflexiona.**
1. El periodo real de las mareas lunares es de unas **12.4 horas**. ¿Lo encontraste?
2. ¿Por qué en las lecciones 3.3–3.5 (la recta de la heladería) nunca tuvimos este problema?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Sí: `f ≈ 0.507` → periodo = 2π / 0.507 ≈ **12.4 h**. El modelo descubrió el ritmo de la Luna a partir de las mediciones… cuando lo dejamos explorar lo suficiente.
2. Porque la recta con MSE es **convexa**: un único tazón. Aquí la perilla está **dentro** de un seno: cambiar `f` desplaza las ondas y crea muchos encajes parciales, es decir, muchos valles. Lo mismo pasa en las redes neuronales: sus perillas están dentro de funciones no lineales (Lección 9.3).
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *distill.pub/2017/momentum* ("Why Momentum Really Works"): juega con el momento en distintos paisajes.
- 📘 **Artículo:** Dauphin et al., *"Identifying and attacking the saddle point problem in high-dimensional non-convex optimization"* (2014): por qué en muchas dimensiones dominan las sillas.
- 📘 **Artículo:** Li et al., *"Visualizing the Loss Landscape of Neural Nets"* (2018): imágenes reales de paisajes de pérdida de redes neuronales.

---
⬅️ **Anterior:** [3.5 Variantes: batch, estocástico y mini-batch](3.5-batch-estocastico-minibatch.ipynb) · ➡️ **Siguiente:** [3.7 Curvas de aprendizaje: leer la historia del entrenamiento](3.7-curvas-de-aprendizaje.ipynb)